# Which unsupervised `density_auto` interpretation resembles the supplied CSV?

Every score here is fitted without the teammate's predictions. Candidate methods are chosen before reading their Task 5 labels; the labels are used only for a fixed-budget set comparison. High agreement does not prove the teammate used that algorithm or that predictions are correct. The lecturer permits unlabeled Task 5 Test for unsupervised fitting, as confirmed by the user.

In [1]:
from pathlib import Path
import sys, json, joblib
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/pipeline/save_model.py").exists())
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
SRC = ROOT / "src"
import numpy as np
import pandas as pd
from scipy.stats import rankdata
from sklearn.mixture import GaussianMixture
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from src.pipeline.random_seed import set_seed
from src.task5.density_auto import DensityAutoDetector
SEED = set_seed()
frames = {}
for data_name in ["eda_original_v2","eda_1_v2"]:
    card = json.loads((SRC / "task5/data/eda" / f"{data_name}.json").read_text(encoding="utf-8"))
    frame = pd.read_parquet(SRC / card["splits"]["test"]["path"])
    assert frame.player_id.is_unique and "is_anomaly" not in frame
    frames[data_name] = frame.drop(columns="player_id")
assert len(frames["eda_original_v2"]) == len(frames["eda_1_v2"]) == 25889
N = len(frames["eda_original_v2"])
dev_idx, _ = train_test_split(np.arange(N), train_size=12000, test_size=4000, random_state=SEED)
scores = {}
def add(name, values):
    values=np.asarray(values,dtype=float)
    assert values.shape==(N,) and np.isfinite(values).all(),name
    scores[name]=values
    print(name, "recorded")

In [2]:
# Frozen saved unsupervised detectors: no new fitting.
saved = [
    "t5_r2_ocsvm_nu10_original_v2", "t5_r2_ocsvm_nu10_engineered_v2",
    "t5_r2_tail_original_v2", "t5_r2_tail_engineered_v2",
    "t5_r2_gmm_original_v2", "t5_r2_gmm_engineered_v2",
    "t5_r2_lof_original_v2", "t5_r2_lof_engineered_v2",
    "t5_r2_iforest_original_v2", "t5_r2_iforest_engineered_v2",
    "t5_r2_pca_original_v2", "t5_r2_pca_engineered_v2",
]
for name in saved:
    card = json.loads((SRC / "task5/save" / f"{name}.json").read_text(encoding="utf-8"))
    model = joblib.load(SRC / card["model_path"])
    add(name, model.anomaly_score(frames[card["data_name"]]))

t5_r2_ocsvm_nu10_original_v2 recorded


t5_r2_ocsvm_nu10_engineered_v2 recorded


t5_r2_tail_original_v2 recorded


t5_r2_tail_engineered_v2 recorded
t5_r2_gmm_original_v2 recorded


t5_r2_gmm_engineered_v2 recorded


t5_r2_lof_original_v2 recorded


t5_r2_lof_engineered_v2 recorded


t5_r2_iforest_original_v2 recorded


t5_r2_iforest_engineered_v2 recorded
t5_r2_pca_original_v2 recorded


t5_r2_pca_engineered_v2 recorded


In [3]:
# Original-feature MLP autoencoder already fitted in the prior hypothesis.
original_ae = joblib.load(SRC / "task5/save/t5_density_auto_original_v1.joblib")
def latent(model, X):
    values = model._transform(X)
    z = values
    for weights, bias in zip(model.autoencoder_.coefs_[:2], model.autoencoder_.intercepts_[:2]):
        z = np.maximum(0, z @ weights + bias)
    return z

def derived_scores(prefix, model, X):
    density, error = model.component_scores(X)
    add(prefix + "_pca_gmm", density)
    add(prefix + "_ae_error", error)
    add(prefix + "_pca_gmm_ae_rank", model.anomaly_score(X))
    z = latent(model, X)
    gm = GaussianMixture(n_components=4,covariance_type="diag",reg_covar=1e-4,
                         max_iter=120,random_state=SEED).fit(z[dev_idx])
    latent_rarity = -gm.score_samples(z)
    add(prefix + "_latent_gmm", latent_rarity)
    neighbors = NearestNeighbors(n_neighbors=51,algorithm="auto",n_jobs=-1).fit(z[dev_idx])
    distance = neighbors.kneighbors(z,return_distance=True)[0][:,-1]
    add(prefix + "_latent_knn50", distance)
    add(prefix + "_latent_gmm_ae_rank", (rankdata(latent_rarity)+rankdata(error))/(2*N))
    add(prefix + "_latent_knn_ae_rank", (rankdata(distance)+rankdata(error))/(2*N))
    p = model.pca_.transform(model._transform(X))
    neighbors_pca = NearestNeighbors(n_neighbors=51,algorithm="auto",n_jobs=-1).fit(p[dev_idx])
    add(prefix + "_pca_knn50", neighbors_pca.kneighbors(p,return_distance=True)[0][:,-1])

derived_scores("original", original_ae, frames["eda_original_v2"])
# Same architecture on the existing engineered EDA set; still no teammate labels in fitting.
engineered_ae = DensityAutoDetector(anomaly_fraction=.08,random_state=SEED,max_iter=40).fit(frames["eda_1_v2"].iloc[dev_idx])
derived_scores("engineered", engineered_ae, frames["eda_1_v2"])

original_pca_gmm recorded
original_ae_error recorded


original_pca_gmm_ae_rank recorded


original_latent_gmm recorded


original_latent_knn50 recorded
original_latent_gmm_ae_rank recorded
original_latent_knn_ae_rank recorded


original_pca_knn50 recorded


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


engineered_pca_gmm recorded
engineered_ae_error recorded


engineered_pca_gmm_ae_rank recorded


engineered_latent_gmm recorded


engineered_latent_knn50 recorded
engineered_latent_gmm_ae_rank recorded
engineered_latent_knn_ae_rank recorded


engineered_pca_knn50 recorded


In [4]:
# Fixed, equal-percentile-rank blends. The Tail + original OCSVM pair was suggested by prior diagnostics and is marked post hoc.
def equal_rank(a,b): return (rankdata(scores[a])+rankdata(scores[b]))/(2*N)
for name,a,b in [
    ("tail_engineered_plus_ocsvm_original_posthoc", "t5_r2_tail_engineered_v2", "t5_r2_ocsvm_nu10_original_v2"),
    ("tail_engineered_plus_original_latent_gmm", "t5_r2_tail_engineered_v2", "original_latent_gmm"),
    ("ocsvm_original_plus_original_latent_gmm", "t5_r2_ocsvm_nu10_original_v2", "original_latent_gmm"),
    ("tail_engineered_plus_engineered_latent_gmm", "t5_r2_tail_engineered_v2", "engineered_latent_gmm"),
]: add(name,equal_rank(a,b))

tail_engineered_plus_ocsvm_original_posthoc recorded
tail_engineered_plus_original_latent_gmm recorded
ocsvm_original_plus_original_latent_gmm recorded
tail_engineered_plus_engineered_latent_gmm recorded


In [5]:
# Read teammate predictions only after all candidate scores have been built.
FRIEND = Path.home() / "Downloads" / "kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_auto_v1.csv"
friend = pd.read_csv(FRIEND)
sample = pd.read_csv(SRC / "submission/sample_submission.csv",usecols=["id"])
assert len(friend)==N and friend.id.equals(sample.id) and set(friend.task5)=={0,1}
y=friend.task5.to_numpy(dtype=np.int8); k=int(y.sum())
rows=[]
for name,score in scores.items():
    chosen=np.lexsort((np.arange(N),-score))[:k]
    overlap=int(y[chosen].sum())
    rows.append({"candidate":name,"matching_alerts":overlap,"friend_alerts":k,
                 "coverage_of_friend_alerts":overlap/k,"positive_set_jaccard":overlap/(2*k-overlap)})
results=pd.DataFrame(rows).sort_values("matching_alerts",ascending=False).reset_index(drop=True)
display(results)
print("Number of candidates",len(results),"best",results.iloc[0].to_dict())

,candidate,matching_alerts,friend_alerts,coverage_of_friend_alerts,positive_set_jaccard
0,tail_engineered_plus_ocsvm_original_posthoc,1334,2046,0.652004,0.483684
1,t5_r2_ocsvm_nu10_original_v2,1313,2046,0.641740,0.472472
2,t5_r2_ocsvm_nu10_engineered_v2,1209,2046,0.590909,0.419355
3,t5_r2_iforest_engineered_v2,1152,2046,0.563050,0.391837
4,tail_engineered_plus_engineered_latent_gmm,1131,2046,0.552786,0.381966
5,t5_r2_tail_engineered_v2,1109,2046,0.542033,0.371773
6,t5_r2_pca_engineered_v2,1105,2046,0.540078,0.369936
7,t5_r2_gmm_engineered_v2,1082,2046,0.528837,0.359468
8,t5_r2_gmm_original_v2,1015,2046,0.496090,0.329867
9,t5_r2_pca_original_v2,982,2046,0.479961,0.315756


Number of candidates 32 best {'candidate': 'tail_engineered_plus_ocsvm_original_posthoc', 'matching_alerts': 1334, 'friend_alerts': 2046, 'coverage_of_friend_alerts': 0.6520039100684262, 'positive_set_jaccard': 0.48368382886149386}


This is a similarity search over a finite set of unsupervised scores. The best row is **not** proof of the teammate's algorithm, and choosing it based on the supplied predictions should not be presented as independent Task 5 validation.